# Expression data from NCBI GEO
Opened from **BNERC**, the Bionitrogen Fixation Research Database (BicBioEng lab, University of South Dakota).

Links each series to GEO2R for interactive differential expression, and downloads its series matrix into a table.

**How to run:** paste the GEO accessions BNERC copied into `RECORDS`, then choose **Runtime → Run all**.

In [ ]:
#@title GEO series to analyze
#@markdown Paste the GEO accessions BNERC copied for you (e.g. `GSE12345, GSE67890`).
RECORDS = "GSE10000"  #@param {type:"string"}
series = [s.split(":")[0].strip() for s in RECORDS.replace("\n", ",").split(",") if s.strip()]
assert series, "Paste at least one GEO accession into RECORDS above."
print(series)

## 1. Open in GEO2R

In [ ]:
from IPython.display import Markdown, display
display(Markdown("\n".join(f"- [{s} in GEO2R](https://www.ncbi.nlm.nih.gov/geo/geo2r/?acc={s})" for s in series)))

## 2. Download series matrices

In [ ]:
import gzip, io, urllib.request
import pandas as pd
tables = {}
for s in series:
    url = f"https://ftp.ncbi.nlm.nih.gov/geo/series/{s[:-3]}nnn/{s}/matrix/{s}_series_matrix.txt.gz"
    try:
        raw = gzip.decompress(urllib.request.urlopen(url).read()).decode("utf-8", "replace")
        body = raw.split("!series_matrix_table_begin")[1].split("!series_matrix_table_end")[0]
        tables[s] = pd.read_csv(io.StringIO(body), sep="\t", index_col=0)
        print(f"✅ {s}: {tables[s].shape[0]} rows × {tables[s].shape[1]} samples")
    except Exception as e:
        print(f"❌ {s}: {e}")
for s, t in tables.items():
    display(t.head())